# 🚀 Stock Price Predictor V4: Cloud Training Pipeline
### 10-Year Intraday Purged Walk-Forward Cross-Validation + 12 Formulaic Alphas

This notebook trains a dual **LightGBM (0.60) + Random Forest (0.40)** ensemble with Platt calibration on **52,000+ twenty-minute bars** (2016–Present Alpaca SIP data).

#### Features Included:
- **31 V3+ Stationary Features**: Gap-adjusted Fractional Differentiation ($d=0.40$), Normalized ATR Barriers, Intraday VWAP Deviations, and SPY Market Beta.
- **12 WorldQuant 101 Formulaic Alphas**: Vectorized Kakushadze alphas (`alpha_001` through `alpha_006`, `alpha_012`, `alpha_014`, `alpha_041`, `alpha_053`, `alpha_054`, `alpha_101`).
- **Marcos López de Prado Deflated Sharpe Ratio (DSR)** & Purged Walk-Forward Embargoed Splits.
- **Environment Version Pinning**: Aligns exact versions (`scikit-learn==1.9.0`, `numpy==2.4.6`, `lightgbm==4.7.0`) to eliminate pickle deserialization warnings.


## Step 1: Install Pinned Dependencies
Run this cell first to ensure your cloud runtime matches the exact library versions used locally.

In [ ]:
!pip install --quiet \
    numpy==2.4.6 \
    scikit-learn==1.9.0 \
    lightgbm==4.7.0 \
    pandas==3.0.3 \
    alpaca-py>=0.30.0 \
    scipy>=1.12.0 \
    joblib==1.5.3 \
    pytz

## Step 2: Configure Alpaca API Credentials
*(Historical SIP data older than 15 minutes is included in Alpaca's **FREE Basic Plan** since 2016).* 

You can either:
1. Add `APCA_API_KEY_ID` and `APCA_API_SECRET_KEY` to Google Colab Secrets (🔑 icon on the left sidebar).
2. Or enter your keys directly in the cell below:

In [ ]:
import os

# Method A: Colab Secrets (Recommended)
try:
    from google.colab import userdata
    os.environ['APCA_API_KEY_ID'] = userdata.get('APCA_API_KEY_ID')
    os.environ['APCA_API_SECRET_KEY'] = userdata.get('APCA_API_SECRET_KEY')
    print("✅ Loaded Alpaca credentials from Colab Secrets!")
except Exception:
    # Method B: Direct Input Fallback
    os.environ['APCA_API_KEY_ID'] = 'YOUR_ALPACA_KEY_ID'
    os.environ['APCA_API_SECRET_KEY'] = 'YOUR_ALPACA_SECRET_KEY'
    print("ℹ️ Please ensure APCA_API_KEY_ID and APCA_API_SECRET_KEY are set above.")

# 0 = 31-feature V3+ model (matches live inference, safe default).
os.environ['INCLUDE_ALPHAS'] = '0'  # set to '1' for the 43-feature V4 model; compare OOS metrics to V3+ before deploying
print(f"Alpha Features Flag: INCLUDE_ALPHAS = {os.environ.get('INCLUDE_ALPHAS')}")

## Step 3: Run Purged Walk-Forward CV Training Pipeline
This step streams 10 years of Alpaca SIP data in RAM, extracts 43 stationary features + alphas, evaluates purged CV folds, fits Platt calibration, and exports the model artifact.

In [ ]:
import colab_cloud_trainer

# Execute end-to-end training
artifact, metadata = colab_cloud_trainer.main()

print("\n🎯 Training Finished!")
print(f"Model Version: {metadata.get('model_version')}")
print(f"Total Features: {len(metadata.get('features', []))}")
print(f"OOS Directional Accuracy: {metadata.get('metrics', {}).get('directional_accuracy', 0):.2%}")

## Step 4: Download Artifact to Local Repository
The training script automatically triggers browser downloads for:
1. `stock_price_model.pkl` (< 5 MB plain-dict artifact)
2. `model_metadata.json`

If auto-download did not trigger, run the cell below:

In [ ]:
try:
    from google.colab import files
    files.download('stock_price_model.pkl')
    files.download('model_metadata.json')
    print("📥 Download initiated!")
except ImportError:
    print("Not in Google Colab environment. Files are saved in the current directory.")